一开始我发现kx地点的P@K图中，P最大只有0.017，最低甚至来到了0这个低谷，于是我想排查看看到底是什么原因。
第一步，排查是不是LabelMe的标签标错了：

In [3]:
import pandas as pd

df = pd.read_csv("outputs/retrieval_results/retrieval_topk.csv")

kx = df[df["query_label"] == "kx"].copy()

# 只看每张 kx query 的 Top20
for qname, group in kx.groupby("query_name"):
    print("\n" + "=" * 80)
    print("Query:", qname)
    print("=" * 80)
    
    show = group.sort_values("rank").head(20)[
        ["rank", "base_name", "base_label", "similarity", "relevant"]
    ]
    
    print(show.to_string(index=False))


Query: kx-1vgyuhy.jpg
 rank                              base_name base_label  similarity  relevant
    1                          yk-xvccad.jpg         yk    0.594312     False
    2                         mh-wer3242.jpg         mh    0.587106     False
    3                      oxford_001709.jpg irrelevant    0.568623     False
    4                 yk-iKYDZ8JuDeegAm7.jpg         yk    0.568408     False
    5                      sy-349138uijr.jpg         sy    0.564911     False
    6                 sjz-k1mOG7KE0fSnzj.jpg        sjz    0.557194     False
    7               fhy-k8j7h6g5f4d3s2a1.jpg        fhy    0.551462     False
    8                  yk-hkauwu23718743.jpg         yk    0.545365     False
    9 ty-a974bccb1721943fab7b3cf99fde62d.jpg         ty    0.545150     False
   10                    magdalen_000632.jpg irrelevant    0.541830     False
   11                          yf-3xs345.jpg         yf    0.534195     False
   12                       sy-fbdgnbfgn.

可以看到对于kx-1vgyuhy.jpg的检索结果中，相似程度排在前列的图片标签都不是kx，这说明了检索函数对kx的特征学习比较弱。

第二步，检查base里有多少张kx图片：

In [2]:
from src.retrieval_utils import list_images, get_landmark_label
from pathlib import Path

base_images = list_images("image_retrieval/base")
query_images = list_images("image_retrieval/query")

print("base kx 数量:", sum(get_landmark_label(p) == "kx" for p in base_images))
print("query kx 数量:", sum(get_landmark_label(p) == "kx" for p in query_images))

base kx 数量: 53
query kx 数量: 2


可以看到确认样中kx的数量比较少。
推测可能因为评价样本过少，导致平均结果很容易被这两张图影响。
如果这两张 query 的角度、颜色、光照和 base 里的 kx 图差异大，或者 kx 和其他地点外观很像，当前 OpenCV 手工特征就容易检索错。

考虑加一个二次排序 rerank：用现在的全局特征先找 Top200，用 ORB 局部特征重新比较这 Top200，再重新排序，输出 Top60。

实践后发现P@K基本没有改变。进而发现yk、yf、nm的P@K也很低。
其中kx和yk在query中的数量都是2，推测可能因为样本过少，所以学习效果不理想；yf可能是因为和jx的外观相似，导致特征检索错误；nm样本不少，外观也比较独特，暂时不清楚为什么P@K格外低。接下来测试ORB对kx的贡献：

In [1]:
import pandas as pd

df = pd.read_csv("outputs/retrieval_results/retrieval_topk.csv")
kx = df[df["query_label"] == "kx"].copy()

for qname, group in kx.groupby("query_name"):
    print("\nQuery:", qname)
    show = group.sort_values("rank").head(20)[
        ["rank", "base_name", "base_label", "global_similarity", "orb_similarity", "similarity", "relevant"]
    ]
    print(show.to_string(index=False))


Query: kx-1vgyuhy.jpg
 rank                              base_name base_label  global_similarity  orb_similarity  similarity  relevant
    1                      oxford_001709.jpg irrelevant           0.568623          0.1125    0.408980     False
    2                          yk-xvccad.jpg         yk           0.594312          0.0500    0.403803     False
    3                         mh-wer3242.jpg         mh           0.587106          0.0625    0.403494     False
    4                      sy-349138uijr.jpg         sy           0.564911          0.1000    0.402192     False
    5                 yk-iKYDZ8JuDeegAm7.jpg         yk           0.568408          0.0875    0.400090     False
    6               fhy-k8j7h6g5f4d3s2a1.jpg        fhy           0.551462          0.1000    0.393451     False
    7 ty-a974bccb1721943fab7b3cf99fde62d.jpg         ty           0.545150          0.0875    0.384972     False
    8                 ty-hhYis0xwbPxsod8.jpg         ty           0.52517

可以看到虽然orb_similarity值并没有出现0.0000，rank排在前面的还是 yk/mh/sy，说明这些错误图片和 query 里面的 kx 在局部纹理上很像，ORB 被误导了。

尝试降低全局颜色/边缘特征权重后，kx的P@K最低值来到了0.025，最高值为0.05，稍有改善但不明显。也许是受限于 OpenCV 手工特征的上限。